# slice-view-mutation — ex1: in-place zero the diagonal via slice-view writes

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `slice-view-mutation`. Running the final beacon cell reports progress against the `PyTorch: Slice view mutation` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Slice view mutation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`slice-view-mutation`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "slice-view-mutation"
DD_SUBTOPIC = "PyTorch: Slice view mutation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Slice view mutation — quick refresher

Slicing a tensor with `:` and integer-range syntax returns a **view** that shares storage with the source. Writes through the view alias the source:
```python
x = t.zeros(4, 4)
row = x[1]           # view — same storage as x
row[:] = 7.0         # mutates x[1] in place
```

**Contrast with boolean / fancy indexing**, which returns a **copy**. If you ever see `x[mask] = value` work but `subset = x[mask]; subset[:] = value` not, this is why — the second form mutates an independent copy.

**`.clone()` breaks the aliasing.** Use it when you want a slice you can modify without affecting the source.

### Exercise 1 — in-place zero the diagonal via slice-view writes

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Mutate a square matrix in place by writing through a slice-view (the diagonal), and verify the source tensor reflects the change because slices return views, not copies.
> Keywords: slice, view, in-place, diagonal
> ```

**KCs targeted:** `slice-returns-view`, `view-writes-alias-source`

Implement `ex1_zero_diagonal_inplace(mat)`.

Given a square `(N, N)` float tensor `mat`, set every diagonal entry to `0.0` **by writing through a slice-view of `mat`** — no `mat = ...` reassignment, no `mat.fill_diagonal_(0)`. The point is to exercise the view-aliasing property.

**Hint.** `mat.diagonal()` (or `mat.diag()`) returns a 1-D view-tensor of length `N` that shares storage with `mat`. Writing `view[:] = 0.0` mutates `mat` in place.

Inputs: `mat` — `(N, N)` float tensor.
Output: the function should **return the same `mat` object** (not a copy). All diagonal entries are now zero; off-diagonal entries are untouched.

In [ ]:
def ex1_zero_diagonal_inplace(mat: Tensor) -> Tensor:
    mat.diagonal()[:] = 0.0
    return mat


<details><summary>Solution</summary>

```python
def ex1_zero_diagonal_inplace(mat: Tensor) -> Tensor:
    mat.diagonal()[:] = 0.0
    return mat
```

**`mat.diagonal()` returns a view.** It's a 1-D tensor of length `N` whose elements alias `mat[0, 0], mat[1, 1], ..., mat[N-1, N-1]`. Writing `[:] = 0.0` through it scatters back to the source.

**Why `mat.diagonal() = 0` would NOT work.** That's a Python rebind of a local name; it doesn't go through `__setitem__` and doesn't mutate `mat`. You need slice-assignment (`[:]` or `[...]`) to trigger the in-place write path.

**Contrast with boolean indexing.** `mat[mat > 5] = 0` works (direct `__setitem__`), but `view = mat[mat > 5]; view[:] = 0` does NOT mutate `mat` — boolean indexing returns a copy, not a view, so the slice-write goes to nowhere useful.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()